# NN_01 — tensor construction

`BUILD_SPEC.md` §7, row 01: one real cycle → the six tensors of `OUTLINE.md` §3.1.

| tensor | shape | contents |
|---|---|---|
| `X` | `(N, 63)` | stock-level continuous features, `NaN` kept where missing |
| `mask` | `(N, 63)` | `True` where the feature was missing |
| `sector` | `(N,)` int64 | index into the 10-category sector vocabulary |
| `quarter` | `(N,)` int64 | fiscal quarter 1–4 |
| `m` | `(6,)` | the cycle's macro vector — **stored once**, never tiled to `(N, 6)` |
| `y` | `(N,)` int64 | relevance grade 0–9 from `labeling.py`, already computed |

The builder lives in `nn_common.py` (`build_cycle_tensors`, `bundle_cycle_tensors`, `to_torch`) because every later notebook needs it identically. It contains no model code and no thresholds.

**Data mode: REAL DATA.** Every tensor below is built from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`), used as-is. No synthetic, simulated, or approximated value appears anywhere; if the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

**Missingness is masked, never imputed** (`BUILD_SPEC.md` §5, rule 4). At this layer `X` keeps `NaN` wherever the pipeline reported the feature missing, and `mask` is `True` there. No value is invented. The encoder (`NN_04`) zero-fills *under the mask* at input time, so a missing entry and a genuine zero remain distinguishable inputs.

**Acceptance for this row, as agreed after `NN_00`:** `X.shape[0] == len(observations)`; `y` within 0..9; and the mask **exactly matches** the pipeline's own per-row `missing_features` — not an approximate 1.03/row, which is a full-history figure this slice does not reproduce. The measured rate is reported as context only.

Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_01_tensor_construction.ipynb")

neural_ranker / NN_01_tensor_construction.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## Feature partition (`BUILD_SPEC.md` §4, used exactly)

63 stock-level continuous + 6 macro + 2 categorical = 71 = `FEATURE_NAMES`. `NEVER_RANK` is a subset of the stock-level block and is consumed by `NN_02`'s policy, not here.

In [2]:
from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES

print("FEATURE_NAMES     ", len(FEATURE_NAMES))
print("STOCK_CONTINUOUS  ", len(nc.STOCK_CONTINUOUS), "-> columns of X and mask, in FEATURE_NAMES order")
print("MACRO             ", len(nc.MACRO), nc.MACRO)
print("CATEGORICAL       ", len(nc.CATEGORICAL), nc.CATEGORICAL)
print("NEVER_RANK        ", len(nc.NEVER_RANK), nc.NEVER_RANK)
print("SECTOR_VOCABULARY ", len(nc.SECTOR_VOCABULARY), nc.SECTOR_VOCABULARY)

partition = nc.STOCK_CONTINUOUS + nc.MACRO + nc.CATEGORICAL
assert (len(nc.STOCK_CONTINUOUS), len(nc.MACRO), len(nc.CATEGORICAL)) == (63, 6, 2)
assert sorted(partition) == sorted(FEATURE_NAMES) and len(set(partition)) == 71, "partition must cover FEATURE_NAMES exactly once"
assert set(nc.NEVER_RANK) <= set(nc.STOCK_CONTINUOUS)
assert set(rd.config().cross_sectional_rank_features) <= set(nc.STOCK_CONTINUOUS)
assert len(nc.SECTOR_VOCABULARY) == 10

FEATURE_NAMES      71
STOCK_CONTINUOUS   63 -> columns of X and mask, in FEATURE_NAMES order
MACRO              6 ('fed_funds_rate', 'hy_credit_oas', 'ust_10y_yield', 'ust_2y_yield', 'vix', 'yield_curve_10y_2y')
CATEGORICAL        2 ('sector_enc', 'quarter_num')
NEVER_RANK         8 ('volatility_20d', 'volatility_30d', 'volatility_63d', 'volatility_90d', 'vol_20d', 'vol_63d', 'vol_ratio', 'beta')
SECTOR_VOCABULARY  10 ('Consumer', 'Energy', 'Financials', 'Health Care', 'Industrials', 'Materials', 'Real Estate', 'Tech & Media', 'Unknown', 'Utilities')


## One real cycle

Selection rule, fixed before looking at any number: **the most recent cycle whose labels have been realized** — the second-to-last cycle of the slice, since the last one has no next quarter to realize a forward return against. That cycle also has a measured baseline IC, so it is one of the 8 cycles `NN_05`–`NN_08` will pair on.

In [3]:
if rd.require_data():
    bundle = rd.slice_bundle()
    CYCLE = bundle.cycles[-2]
    observations = bundle.feature_results[CYCLE.quarter_start].observations
    labeled = bundle.labeled_quarters[CYCLE.quarter_start]
    baseline = next(r for r in bundle.ic_result.cycle_results if r.cycle.quarter_start == CYCLE.quarter_start)

    print("cycle quarter_start ", CYCLE.quarter_start)
    print("cycle cutoff        ", CYCLE.cutoff, "(only data with available_date <= cutoff entered these observations)")
    print("observations        ", len(observations))
    print("labeled rows        ", len(labeled))
    print("baseline IC         ", f"{baseline.ic:+.4f}", "over", baseline.scored_for_ic_count, "scored names")

    T = nc.build_cycle_tensors(CYCLE, observations, labeled)
    print()
    print(f"{'tensor':<8}{'shape':<14}{'dtype':<10}note")
    print(f"{'X':<8}{str(T.X.shape):<14}{str(T.X.dtype):<10}NaN kept where missing")
    print(f"{'mask':<8}{str(T.mask.shape):<14}{str(T.mask.dtype):<10}True where missing")
    print(f"{'sector':<8}{str(T.sector.shape):<14}{str(T.sector.dtype):<10}index into SECTOR_VOCABULARY")
    print(f"{'quarter':<8}{str(T.quarter.shape):<14}{str(T.quarter.dtype):<10}fiscal quarter 1..4")
    print(f"{'m':<8}{str(T.m.shape):<14}{str(T.m.dtype):<10}one macro vector for the whole cycle")
    print(f"{'y':<8}{str(T.y.shape):<14}{str(T.y.dtype):<10}relevance grade, available at {T.label_available_at.date()}")

    # Acceptance (BUILD_SPEC §7, row 01)
    assert T.X.shape[0] == len(observations)
    assert T.X.shape == T.mask.shape == (len(observations), 63)
    assert T.sector.shape == T.quarter.shape == T.y.shape == (len(observations),)
    assert T.m.shape == (6,)
    assert T.instrument_ids == tuple(o.instrument_id for o in observations), "row order is the cross-section's own order"

cycle quarter_start  2023-10-01
cycle cutoff         2023-09-30 (only data with available_date <= cutoff entered these observations)
observations         1493
labeled rows         1493
baseline IC          +0.1240 over 1401 scored names

tensor  shape         dtype     note
X       (1493, 63)    float64   NaN kept where missing
mask    (1493, 63)    bool      True where missing
sector  (1493,)       int64     index into SECTOR_VOCABULARY
quarter (1493,)       int64     fiscal quarter 1..4
m       (6,)          float64   one macro vector for the whole cycle
y       (1493,)       int64     relevance grade, available at 2024-01-01


## The mask exactly matches `missing_features`

The builder already refuses a mismatch, but this cell recomputes the check independently: for every row, the set of stock-level names the pipeline listed in `missing_features` must equal the set of columns where `mask` is `True`, and the number of `NaN`s in `X` must equal the number of `True`s in `mask`. It also checks that no macro or categorical feature is missing on any row, so the 71-wide `missing_features` count and the 63-wide mask count are the same number.

In [4]:
if rd.require_data():
    import numpy as np

    stock = set(nc.STOCK_CONTINUOUS)
    mismatches = 0
    reported_71 = 0
    reported_63 = 0
    for i, obs in enumerate(observations):
        reported = {name for name in obs.missing_features if name in stock}
        found = {name for name, is_missing in zip(nc.STOCK_CONTINUOUS, T.mask[i]) if is_missing}
        mismatches += reported != found
        reported_71 += len(obs.missing_features)
        reported_63 += len(reported)
    non_stock_missing = reported_71 - reported_63

    N = T.n
    print(f"rows checked                         {N:,}")
    print(f"rows where mask != missing_features  {mismatches}")
    print(f"NaN entries in X                     {int(np.isnan(T.X).sum()):,}")
    print(f"True entries in mask                 {int(T.mask.sum()):,}")
    print(f"missing_features entries (71-wide)   {reported_71:,}")
    print(f"  of which macro/categorical         {non_stock_missing}")
    print(f"rows with >= 1 missing feature       {int(T.mask.any(axis=1).sum()):,}  ({T.mask.any(axis=1).mean():.1%})")
    print(f"max missing in one row               {int(T.mask.sum(axis=1).max())}")
    fully_masked = np.where(T.mask.all(axis=1))[0]
    print(f"rows with all 63 missing             {len(fully_masked)}  "
          f"{[T.instrument_ids[i].symbol for i in fully_masked]}  (kept; they carry only sector, quarter and the mask)")
    print()
    print(f"measured missing per row, this cycle {T.mask.sum() / N:.3f}   (context: BUILD_SPEC §3 quotes 1.03 for the full history; not asserted)")

    assert mismatches == 0, "mask must equal missing_features on every row"
    assert int(np.isnan(T.X).sum()) == int(T.mask.sum()) == reported_63
    assert non_stock_missing == 0, "a macro or categorical feature is missing -- would need explicit handling"

    per_feature = T.mask.sum(axis=0)
    order = np.argsort(-per_feature)
    print()
    print("most-missing features this cycle:")
    for j in order[:8]:
        print(f"  {nc.STOCK_CONTINUOUS[j]:<40}{int(per_feature[j]):>5}  ({per_feature[j] / N:.1%})")

rows checked                         1,493
rows where mask != missing_features  0
NaN entries in X                     1,542
True entries in mask                 1,542
missing_features entries (71-wide)   1,542
  of which macro/categorical         0
rows with >= 1 missing feature       315  (21.1%)
max missing in one row               63
rows with all 63 missing             1  ['VSTS']  (kept; they carry only sector, quarter and the mask)

measured missing per row, this cycle 1.033   (context: BUILD_SPEC §3 quotes 1.03 for the full history; not asserted)

most-missing features this cycle:
  pe_ratio                                  178  (11.9%)
  price_to_book                              99  (6.6%)
  price_mom_12m                              50  (3.3%)
  vol_63d                                    49  (3.3%)
  vol_ratio                                  49  (3.3%)
  price_to_sales                             47  (3.1%)
  price_mom_6m                               44  (2.9%)
  beta     

## Every cycle of the slice

The same construction over all 14 cycles, with the exact-match check repeated on each. This is where the measured missing rate is reported honestly: it is a per-cycle quantity that ranges well beyond the single full-history figure.

In [5]:
if rd.require_data():
    ALL = nc.bundle_cycle_tensors(bundle)

    print(f"{'quarter_start':<14}{'N':>6}{'missing/row':>13}{'rows w/ missing':>17}{'labels':>8}{'label_available_at':>20}")
    total_missing = total_rows = 0
    for cycle in bundle.cycles:
        t = ALL[cycle.quarter_start]
        obs_cycle = bundle.feature_results[cycle.quarter_start].observations
        # exact-match check, independently of the builder's own guard
        for i, obs in enumerate(obs_cycle):
            assert {n for n in obs.missing_features if n in stock} == {
                n for n, miss in zip(nc.STOCK_CONTINUOUS, t.mask[i]) if miss
            }
            assert len(obs.missing_features) == int(t.mask[i].sum())
        total_missing += int(t.mask.sum())
        total_rows += t.n
        print(
            f"{cycle.quarter_start.isoformat():<14}{t.n:>6}{t.mask.sum() / t.n:>13.3f}"
            f"{t.mask.any(axis=1).mean():>16.1%}{('yes' if t.y is not None else 'no'):>8}"
            f"{(t.label_available_at.date().isoformat() if t.label_available_at else '-'):>20}"
        )
    rates = [ALL[c.quarter_start].mask.sum() / ALL[c.quarter_start].n for c in bundle.cycles]
    fully_masked = [
        (c.quarter_start, ALL[c.quarter_start].instrument_ids[i].symbol)
        for c in bundle.cycles
        for i in np.where(ALL[c.quarter_start].mask.all(axis=1))[0]
    ]
    print()
    print(f"cycles built                 {len(ALL)}")
    print(f"rows built                   {total_rows:,}")
    print(f"missing per row, whole slice {total_missing / total_rows:.3f}   (range {min(rates):.3f} .. {max(rates):.3f} across cycles)")
    print(f"rows with all 63 missing     {len(fully_masked)}   {[(q.isoformat(), s) for q, s in fully_masked]}")
    print("context only: BUILD_SPEC §3's 1.03/row is a full-history mean and is not reproduced by this slice")

    assert len(ALL) == len(bundle.cycles) == 14
    assert sum(1 for t in ALL.values() if t.y is not None) == 13 and ALL[bundle.cycles[-1].quarter_start].y is None

quarter_start      N  missing/row  rows w/ missing  labels  label_available_at
2020-10-01      1423        1.552           25.1%     yes          2021-01-01
2021-01-01      1430        1.592           27.7%     yes          2021-04-01
2021-04-01      1436        1.428           27.4%     yes          2021-07-01
2021-07-01      1451        1.606           27.8%     yes          2021-10-01
2021-10-01      1462        1.576           27.6%     yes          2022-01-01
2022-01-01      1468        1.402           26.1%     yes          2022-04-01
2022-04-01      1469        1.295           24.8%     yes          2022-07-01
2022-07-01      1472        1.127           23.6%     yes          2022-10-01
2022-10-01      1474        0.988           23.1%     yes          2023-01-01
2023-01-01      1478        0.989           23.5%     yes          2023-04-01
2023-04-01      1479        0.965           22.5%     yes          2023-07-01
2023-07-01      1488        1.048           22.2%     yes      

## Categorical tensors: `sector` and `quarter`

`sector_enc` is already the index into `SectorEncoder().consolidated_vocabulary()`; the builder verifies that the index names each observation's own `sector` string. Note that **Materials rows are present in the features**: the strategy's `exclude_sectors=("Materials",)` is applied at ranking time by the decision pipeline, so the baseline's per-cycle IC is measured over the *ranked* names only. `NN_05` must score the NN over the same names the baseline was measured on before pairing.

In [6]:
if rd.require_data():
    counts = np.bincount(T.sector, minlength=len(nc.SECTOR_VOCABULARY))
    print(f"{'index':<7}{'sector':<16}{'count':>6}")
    for idx, (name, count) in enumerate(zip(nc.SECTOR_VOCABULARY, counts)):
        print(f"{idx:<7}{name:<16}{int(count):>6}")
    print()
    q_counts = {q: int((T.quarter == q).sum()) for q in (1, 2, 3, 4)}
    print("fiscal quarter of the selected filing:", q_counts)

    assert T.sector.min() >= 0 and T.sector.max() < len(nc.SECTOR_VOCABULARY)
    assert all(nc.SECTOR_VOCABULARY[s] == obs.sector for s, obs in zip(T.sector, observations))
    assert T.quarter.min() >= 1 and T.quarter.max() <= 4
    materials = int(counts[nc.SECTOR_VOCABULARY.index("Materials")])
    print(f"\nMaterials rows in the features: {materials}  (excluded only at ranking time; baseline IC scored {baseline.scored_for_ic_count} names)")

index  sector           count
0      Consumer           270
1      Energy              71
2      Financials         261
3      Health Care        161
4      Industrials        257
5      Materials           73
6      Real Estate         97
7      Tech & Media       240
8      Unknown              1
9      Utilities           62

fiscal quarter of the selected filing: {1: 0, 2: 1254, 3: 238, 4: 1}

Materials rows in the features: 73  (excluded only at ranking time; baseline IC scored 1401 names)


## The macro vector `m (6,)`

Stored **once per cycle**. The builder requires every macro feature to take exactly one non-`NaN` value across the whole cross-section — the broadcast the pipeline performs at the cycle's cutoff — and refuses to build otherwise. The table over all 14 cycles is the real range of macro states the FiLM conditioner will see in this slice.

In [7]:
if rd.require_data():
    print(f"m for {CYCLE.quarter_start}:")
    for name, value in zip(T.macro_names, T.m):
        print(f"  {name:<20}{value:>10.4f}")

    # Independent check: every macro feature has exactly one distinct value over the N rows.
    for j, name in enumerate(nc.MACRO):
        column = np.array([obs.features[name] for obs in observations])
        assert not np.isnan(column).any() and len(np.unique(column)) == 1 and column[0] == T.m[j]

    print()
    print(f"{'quarter_start':<14}" + "".join(f"{n:>20}" for n in nc.MACRO))
    for cycle in bundle.cycles:
        t = ALL[cycle.quarter_start]
        print(f"{cycle.quarter_start.isoformat():<14}" + "".join(f"{v:>20.4f}" for v in t.m))
    M = np.stack([ALL[c.quarter_start].m for c in bundle.cycles])
    print()
    print("range across the slice:")
    for j, name in enumerate(nc.MACRO):
        print(f"  {name:<20}{M[:, j].min():>10.4f} .. {M[:, j].max():>8.4f}")
    assert M.shape == (14, 6) and not np.isnan(M).any()

m for 2023-10-01:
  fed_funds_rate          5.3300
  hy_credit_oas           3.9400
  ust_10y_yield           4.5711
  ust_2y_yield            5.0436
  vix                    17.5200
  yield_curve_10y_2y     -0.4725

quarter_start       fed_funds_rate       hy_credit_oas       ust_10y_yield        ust_2y_yield                 vix  yield_curve_10y_2y
2020-10-01                  0.0900              5.1700              0.6840              0.1270             26.3700              0.5570
2021-01-01                  0.0900              3.6000              0.9132              0.1211             22.7500              0.7921
2021-04-01                  0.0600              3.1000              1.7404              0.1603             19.4000              1.5801
2021-07-01                  0.0800              2.6800              1.4680              0.2486             15.8300              1.2194
2021-10-01                  0.0600              2.8900              1.4873              0.2755             2

## The relevance target `y`

Grades come from `labeling.py` via `LabeledObservation.relevance` — already computed, reused as-is (`BUILD_SPEC.md` §2). Grade 9 is the best-performing decile of realized forward return, grade 0 the worst. Grade 0 also receives every instrument with **no computable forward return** (excluded from the ranking but kept in the training set), which is why it is larger than the other nine — the same treatment the baseline trains on, so it is kept identical here.

In [8]:
if rd.require_data():
    hist = np.bincount(T.y, minlength=10)
    print("grade   count")
    for grade, count in enumerate(hist):
        print(f"  {grade}     {int(count):>5}  {'#' * (int(count) // 5)}")
    print()
    print("median count over grades 1..9 ", int(np.median(hist[1:])))
    print("excess in grade 0             ", int(hist[0] - np.median(hist[1:])), "(consistent with the no-computable-return rule; the slice does not store the exact excluded count)")
    print("label_available_at            ", T.label_available_at, "> entry", CYCLE.quarter_start, "=", T.label_available_at.date() > CYCLE.quarter_start)

    # Acceptance (BUILD_SPEC §7, row 01): y within 0..9.
    assert T.y.min() >= 0 and T.y.max() <= 9
    assert len(np.unique(T.y)) == 10
    # Alignment: y[i] is the grade of the instrument in row i, looked up independently.
    by_id = {row.observation.instrument_id: row.relevance for row in labeled}
    assert all(by_id[obs.instrument_id] == grade for obs, grade in zip(observations, T.y))

grade   count
  0       167  #################################
  1       147  #############################
  2       147  #############################
  3       148  #############################
  4       147  #############################
  5       147  #############################
  6       148  #############################
  7       147  #############################
  8       147  #############################
  9       148  #############################

median count over grades 1..9  147
excess in grade 0              20 (consistent with the no-computable-return rule; the slice does not store the exact excluded count)
label_available_at             2024-01-01 00:00:00 > entry 2023-10-01 = True


## `X` at a glance, raw scale

Non-missing count and range of each of the 63 columns for this cycle, before any policy from `NN_02`. Two things to see: the five features the pipeline already percentile-ranks arrive exactly in `[0, 1]`, and the rest span many orders of magnitude (basis-point changes up to ~10⁶, growth rates in the hundreds), which is the scaling problem `NN_02` exists to address.

In [9]:
if rd.require_data():
    ranked_by_pipeline = set(rd.config().cross_sectional_rank_features)
    print(f"{'feature':<42}{'non-missing':>12}{'min':>14}{'median':>12}{'max':>14}  note")
    for j, name in enumerate(nc.STOCK_CONTINUOUS):
        column = T.X[:, j]
        ok = ~np.isnan(column)
        note = "pipeline percentile rank" if name in ranked_by_pipeline else ("NEVER_RANK" if name in nc.NEVER_RANK else "")
        print(
            f"{name:<42}{int(ok.sum()):>12}{np.nanmin(column):>14.4g}{np.nanmedian(column):>12.4g}"
            f"{np.nanmax(column):>14.4g}  {note}"
        )
    for name in ranked_by_pipeline:
        column = T.X[:, nc.STOCK_CONTINUOUS.index(name)]
        assert np.nanmin(column) == 0.0 and np.nanmax(column) == 1.0, f"{name} should already be a [0, 1] percentile rank"

feature                                    non-missing           min      median           max  note
market_cap                                        1464             0         0.5             1  pipeline percentile rank
volatility_30d                                    1455         9.551       29.48         207.3  NEVER_RANK
analyst_target_price                              1453             0      0.4997             1  pipeline percentile rank
pe_ratio                                          1315        0.3322       19.71          1214  
price_to_book                                     1394        0.1988       2.586          1219  
volatility_63d                                    1453         10.12       30.63         173.2  NEVER_RANK
volatility_20d                                    1457         7.562       27.02           233  NEVER_RANK
volume                                            1460             0         0.5             1  pipeline percentile rank
price_to_sales       

## To `torch`, on the device

MPS has no float64, so `to_torch` casts floats to float32 and leaves integer tensors as int64. The `NaN`s are still there under the mask — nothing is zero-filled at this layer — and `m` moves as a single `(6,)` row.

In [10]:
if rd.require_data():
    import torch

    DEVICE = nc.resolve_device()
    D = nc.to_torch(T, DEVICE)
    print("device", DEVICE)
    for key, tensor in D.items():
        print(f"  {key:<8}{str(tuple(tensor.shape)):<14}{str(tensor.dtype):<15}{tensor.device}")

    assert all(tensor.device.type == DEVICE.type for tensor in D.values())
    assert D["X"].shape == (T.n, 63) and D["mask"].shape == (T.n, 63) and D["m"].shape == (6,)
    assert D["X"].dtype == D["mask"].dtype == D["m"].dtype == torch.float32
    assert D["sector"].dtype == D["quarter"].dtype == D["y"].dtype == torch.int64
    nan_count = int(torch.isnan(D["X"]).sum().item())
    print(f"\nNaN entries in X on {DEVICE}: {nan_count}  == mask sum {int(D['mask'].sum().item())}: {nan_count == int(D['mask'].sum().item())}")
    assert nan_count == int(D["mask"].sum().item()) == int(T.mask.sum())
    assert torch.equal(D["mask"], torch.isnan(D["X"]).float())

device mps
  X       (1493, 63)    torch.float32  mps:0
  mask    (1493, 63)    torch.float32  mps:0
  sector  (1493,)       torch.int64    mps:0
  quarter (1493,)       torch.int64    mps:0
  m       (6,)          torch.float32  mps:0
  y       (1493,)       torch.int64    mps:0

NaN entries in X on mps: 1542  == mask sum 1542: True


## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6. Tensor construction computes no statistic across rows or across cycles — every entry of `X`, `mask`, `sector`, `quarter` is a per-row copy of one observation's own value, and `m` is the one value the pipeline broadcast at this cycle's cutoff. What was checked here, not assumed:

1. **The builder can only see one cycle.** Its signature admits one cycle's observations and that cycle's labels; the cell below shows it *refuses* observations carrying another cycle's cutoff.
2. **Nothing in `X` derives from `y`.** `X`'s columns are exactly `STOCK_CONTINUOUS`; the relevance grade is carried separately and is used only as a training target for *later* cycles (`NN_05`, through `training_dataset.build_training_dataset`, which excludes the target quarter and everything after it).
3. **`y` is not knowable at the cycle's own entry.** `label_available_at` is the next quarter's start, strictly after `quarter_start`; the builder enforces this.
4. **Independence from other cycles at this layer.** Building the cycle in isolation, and building it as part of the full 14-cycle pass, yields byte-identical tensors. (`NN_03` extends this to the normalized tensors, which is where cross-row statistics first appear.)

In [11]:
if rd.require_data():
    from datetime import datetime, time as _time

    # 1. refuses another cycle's observations
    other = bundle.cycles[-1]
    try:
        nc.build_cycle_tensors(CYCLE, bundle.feature_results[other.quarter_start].observations, None)
        refused = False
    except ValueError as exc:
        refused = True
        print("refused foreign observations:", str(exc)[:110], "...")
    assert refused

    # 2. X carries no label-derived column
    assert T.feature_names == nc.STOCK_CONTINUOUS and "relevance" not in T.feature_names

    # 3. y not knowable at entry
    assert T.label_available_at > datetime.combine(CYCLE.quarter_start, _time.min)

    # 4. byte-identical whether built alone or in the full pass
    alone = nc.build_cycle_tensors(CYCLE, observations, labeled)
    in_pass = ALL[CYCLE.quarter_start]
    identical = all(
        getattr(alone, k).tobytes() == getattr(in_pass, k).tobytes() == getattr(T, k).tobytes()
        for k in ("X", "mask", "sector", "quarter", "m", "y")
    )
    print("byte-identical alone vs full pass:", identical)
    assert identical
    print("point-in-time checks 1-4: passed")

refused foreign observations: cycle 2023-10-01: 1494 observation(s) carry a data_cutoff other than this cycle's 2023-09-30 (first: A @ 2023- ...
byte-identical alone vs full pass: True
point-in-time checks 1-4: passed


## What this established

Every number below was printed by a cell above in this executed run.

**The six tensors exist for a real cycle.** Cycle 2023-10-01 (cutoff 2023-09-30, the most recent cycle with realized labels): `X (1493, 63)` float64, `mask (1493, 63)` bool, `sector (1493,)` int64, `quarter (1493,)` int64, `m (6,)` float64, `y (1493,)` int64. `X.shape[0] == len(observations) == 1493`; rows are in the cross-section's own order. Its baseline IC is +0.1240 over 1,401 scored names, and its labels became available on 2024-01-01.

**The mask exactly matches the pipeline's `missing_features`.** 0 mismatching rows out of 1,493; 1,542 `NaN`s in `X` = 1,542 `True`s in `mask` = 1,542 `missing_features` entries, none of them macro or categorical. 315 rows (21.1%) have at least one missing feature; one row (`VSTS`) has all 63 missing. The same exact-match check passed on every row of all 14 cycles (20,517 rows).

**Measured missing rate, reported as context and not asserted.** 1.033 per row for this cycle; **1.252 per row over the whole slice**, ranging from 0.965 (2023-04-01) to 1.606 (2021-07-01) by cycle. `BUILD_SPEC.md` §3's 1.03 is a full-history mean and this slice does not reproduce it. `pe_ratio` (178 rows, 11.9%) and `price_to_book` (99, 6.6%) dominate the missingness. Five rows in the slice have every stock-level feature missing (`NTST`, `STEP` in 2020-10; `LIF` in 2022-07 and 2023-04; `VSTS` in 2023-10); they are kept, carrying only sector, quarter and the mask.

**Categoricals are consistent.** Every `sector_enc` indexes the 10-name vocabulary and names its own observation's `sector`; 73 Materials rows and 1 Unknown row are present in the features (Materials is excluded only at ranking time, so `NN_05` must score the same names the baseline was measured on). Fiscal `quarter_num` is heavily skewed within a cycle (1,254 Q2, 238 Q3, 1 Q4, 0 Q1 as of 2023-09-30) because it is the quarter of each issuer's latest filing, not the calendar quarter.

**`m` is one vector per cycle.** Every macro feature has exactly one non-`NaN` value across the cross-section in all 14 cycles. For 2023-10-01: fed funds 5.33, HY OAS 3.94, 10y 4.5711, 2y 5.0436, VIX 17.52, curve −0.4725. Across the slice: fed funds 0.06 → 5.33, curve +1.58 → −1.06, VIX 12.45 → 31.62 — the whole 2022–23 hiking cycle. The structural caveat this creates for the experiment: there are only **14 distinct macro states** in the slice, and by the training-window rule (a quarter's labels must be available before the target's cutoff) a cycle's model sees between 4 and 12 of them. `NN_04`–`NN_06` must be read against that; `NN_05` will confirm the exact windows.

**`y` is a valid grade vector.** Every value in 0..9, all ten grades present; grade 0 holds 167 rows against a median of 147 for grades 1–9 (excess 20, the instruments with no computable forward return, graded 0 by `labeling.py` — the same rows the baseline trains on). `y[i]` was independently confirmed to be the grade of the instrument in row `i`.

**Raw scale.** The five pipeline-ranked features arrive exactly in `[0, 1]` (asserted). The rest span up to ~1.5 × 10⁶ (basis-point margin changes) and growth rates in the thousands; that scaling is `NN_02`'s job.

**Device.** On `mps`, floats are float32 and integers int64; the 1,542 `NaN`s are still in `X` under the mask — nothing was zero-filled at this layer — and `m` moved as a single `(6,)` row.

**Point-in-time, checked.** The builder refused observations carrying another cycle's cutoff; `X` has no label-derived column; `label_available_at` is strictly after the cycle's entry; and building the cycle alone or inside the 14-cycle pass gives byte-identical tensors. No cross-row or cross-cycle statistic is computed here, so this notebook introduces no lookahead path of its own; `NN_03` audits the normalized tensors, where cross-row statistics first appear.

**Pairing basis for `NN_05`–`NN_08`, stated now.** The cached slice provides **8 measured IC cycles** (2022-01-01 .. 2023-10-01) to pair against. That is thinner than the 22-cycle full-history baseline `BUILD_SPEC.md` §9 quotes, and any p-value from 8 paired cycles has correspondingly little power. This is what the slice holds and it is used as-is.

**What `NN_02` needs from here.** `nc.bundle_cycle_tensors(bundle, transform=...)` accepts a per-cycle observation transform, so the normalization policy is applied to one cross-section at a time and never sees another cycle; `nc.NEVER_RANK` (8 features) and `nc.MACRO` are in place; `apply_cross_sectional_rank_normalization(observations, feature_names)` is the only rank transform to call; and the five already-ranked features arrive in `[0, 1]`, so `NN_02` must decide (and verify, not assume) whether re-ranking them is a no-op.